Import data from 2 key boroughs 

In [9]:
from pathlib import Path
import pandas as pd

# this finds the project root regardless of where the notebook is
ROOT     = Path.cwd().parent
SAMPLES_DIR = ROOT / "data" / "samples"
#Load the datasets
manhattan_sample_df        = pd.read_csv(SAMPLES_DIR / "manhattan_sample.csv")
brooklyn_sample_df = pd.read_csv(SAMPLES_DIR / "brooklyn_sample.csv")    

In [2]:
!pip install transformers torch scipy

  Using cached torch-2.8.0-cp39-cp39-win_amd64.whl (241.2 MB)
  Using cached networkx-3.2.1-py3-none-any.whl (1.6 MB)
  Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
  Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)


You should consider upgrading via the 'C:\Users\avant\source\airbnb-reviews-nlp\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [3]:
import transformers
import torch
import scipy

print(f"transformers=={transformers.__version__}")
print(f"torch=={torch.__version__}")
print(f"scipy=={scipy.__version__}")

c:\Users\avant\source\airbnb-reviews-nlp\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers==4.49.0
torch==2.8.0+cpu
scipy==1.13.1


In [ ]:
# Only needed if running in Colab
import sys
if 'google.colab' in sys.modules:
    !pip install transformers scipy

In [4]:
# ============================================================
# 0. INSTALL & IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
import time
import os

# ============================================================
# 1. DEVICE SETUP
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
# Local: will print 'cpu'
# Colab with GPU: will print 'cuda'


Using device: cpu


In [5]:
# ============================================================
# 2. LOAD MODEL
# ============================================================

MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

print(f"Loading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()

# Labels: 0=Negative, 1=Neutral, 2=Positive
labels = ['Negative', 'Neutral', 'Positive']
print(f"Labels: {labels}")

Loading model: cardiffnlp/twitter-roberta-base-sentiment-latest


c:\Users\avant\source\airbnb-reviews-nlp\.venv\lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\avant\.cache\huggingface\hub\models--cardiffnlp--twitter-roberta-base-sentiment-latest. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-s

Labels: ['Negative', 'Neutral', 'Positive']


In [17]:
# ============================================================
# 3. INFERENCE FUNCTION
# ============================================================

def get_sentiment_batch(texts, batch_size=32):
    """
    Runs sentiment analysis on a list of texts in batches.
    Returns list of dicts with label + confidence scores.
    
    Args:
        texts       : list of review strings
        batch_size  : 32 works well on Colab GPU, reduce to 8-16 for local CPU
    
    Returns:
        List of dicts: [{label, score_negative, score_neutral, score_positive}]
    """
    results = []
    total_batches = len(texts) // batch_size + 1
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        
        # Tokenize — truncate to 512 tokens (model max)
        encoded = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(device)
        
        with torch.no_grad():
            output = model(**encoded)
        
        # Convert logits to probabilities
        scores = softmax(output.logits.cpu().numpy(), axis=1)
        
        for score in scores:
            results.append({
                'sentiment_label'    : labels[np.argmax(score)],
                'score_negative'     : round(float(score[0]), 4),
                'score_neutral'      : round(float(score[1]), 4),
                'score_positive'     : round(float(score[2]), 4),
                'sentiment_confidence': round(float(np.max(score)), 4)
            })
        
        # Progress logging every 10 batches
        if (i // batch_size) % 10 == 0:
            print(f"  Batch {i // batch_size + 1}/{total_batches} — {i + len(batch_texts)} reviews processed")
    
    return results


In [18]:
# ============================================================
# 4. PREPROCESSING
# ============================================================

def preprocess_reviews(df, text_col='reviews'):
    """Clean reviews before passing to model."""
    df = df.copy()
    df[text_col] = df[text_col].astype(str).str.strip()
    
    # Drop empty reviews
    df = df[df[text_col].str.len() > 0]
    df = df[df[text_col] != 'nan']
    
    return df.reset_index(drop=True)

In [19]:

# ============================================================
# 5. RUN ANALYSIS
# ============================================================

def run_transformer_analysis(df, borough_name, text_col='reviews', batch_size=32):
    """
    Full pipeline: preprocess → inference → attach results to df.
    
    Args:
        df          : Borough sample DataFrame
        borough_name: For logging
        text_col    : Column containing review text
        batch_size  : 32 for GPU, 8 for local CPU testing
    
    Returns:
        DataFrame with sentiment columns added
    """
    
    print(f"\n{'='*50}")
    print(f"Running transformer analysis: {borough_name}")
    print(f"{'='*50}")
    
    df = preprocess_reviews(df, text_col)
    print(f"Reviews after preprocessing: {len(df):,}")
    
    texts = df[text_col].tolist()
    
    start = time.time()
    sentiment_results = get_sentiment_batch(texts, batch_size=batch_size)
    elapsed = time.time() - start
    
    print(f"\nInference complete in {elapsed:.1f}s ({elapsed/len(texts)*1000:.1f}ms per review)")
    
    # Attach results
    results_df = pd.DataFrame(sentiment_results)
    df = pd.concat([df.reset_index(drop=True), results_df], axis=1)
    
    # Summary
    print(f"\nSentiment distribution:")
    print(df['sentiment_label'].value_counts(normalize=True).round(3))
    
    return df

In [11]:
manhattan_sample_df.head()

,listing_id,review_id,date,reviews,neighbourhood_group,room_type,price,review_length
0,6990,49986201,2015-10-08,"Cynthia is a very nice host , hospitable and s...",Manhattan,Private room,70.0,59
1,6990,19236,2009-12-05,What can I say about Cynthia?\r<br/>We stayed ...,Manhattan,Private room,70.0,143
2,6990,56559971,2015-12-14,Cynthia is: \r<br/>HONEST\r<br/>KIND\r<br/>HIG...,Manhattan,Private room,70.0,11
3,6990,538492,2011-09-17,In one word - WOW! I really had the time of my...,Manhattan,Private room,70.0,92
4,6990,12175934,2014-04-25,"Being a first time Airbnb user, I had a hard t...",Manhattan,Private room,70.0,186


In [13]:
manhattan_sample_df.columns

Index(['listing_id', 'review_id', 'date', 'reviews', 'neighbourhood_group',
       'room_type', 'price', 'review_length'],
      dtype='object')

In [20]:

# ============================================================
# 6. LOCAL TEST RUN (small sample)
# ============================================================

# Test on 100 rows locally before full Colab run
BATCH_SIZE = 8   # small for CPU — change to 32 on Colab GPU

manhattan_test = manhattan_sample_df.head(100).copy()
manhattan_test_results = run_transformer_analysis(
    manhattan_test, 
    borough_name='Manhattan (test)',
    batch_size=BATCH_SIZE
)

print("\nSample output:")
print(manhattan_test_results[['listing_id', 'reviews', 'sentiment_label', 
                               'score_positive', 'score_neutral', 
                               'score_negative', 'sentiment_confidence']].head(5))


Running transformer analysis: Manhattan (test)
Reviews after preprocessing: 100
  Batch 1/13 — 8 reviews processed
  Batch 11/13 — 88 reviews processed

Inference complete in 37.0s (369.7ms per review)

Sentiment distribution:
sentiment_label
Positive    0.87
Neutral     0.11
Negative    0.02
Name: proportion, dtype: float64

Sample output:
   listing_id                                            reviews  \
0        6990  Cynthia is a very nice host , hospitable and s...   
1        6990  What can I say about Cynthia?\r<br/>We stayed ...   
2        6990  Cynthia is: \r<br/>HONEST\r<br/>KIND\r<br/>HIG...   
3        6990  In one word - WOW! I really had the time of my...   
4        6990  Being a first time Airbnb user, I had a hard t...   

  sentiment_label  score_positive  score_neutral  score_negative  \
0        Positive          0.9820         0.0136          0.0045   
1        Positive          0.9775         0.0184          0.0041   
2         Neutral          0.4533         0

In [21]:
manhattan_test_results.head()

,listing_id,review_id,date,reviews,neighbourhood_group,room_type,price,review_length,sentiment_label,score_negative,score_neutral,score_positive,sentiment_confidence
0,6990,49986201,2015-10-08,"Cynthia is a very nice host , hospitable and s...",Manhattan,Private room,70.0,59,Positive,0.0045,0.0136,0.9820,0.9820
1,6990,19236,2009-12-05,What can I say about Cynthia?\r<br/>We stayed ...,Manhattan,Private room,70.0,143,Positive,0.0041,0.0184,0.9775,0.9775
2,6990,56559971,2015-12-14,Cynthia is: \r<br/>HONEST\r<br/>KIND\r<br/>HIG...,Manhattan,Private room,70.0,11,Neutral,0.0747,0.4720,0.4533,0.4720
3,6990,538492,2011-09-17,In one word - WOW! I really had the time of my...,Manhattan,Private room,70.0,92,Positive,0.0032,0.0094,0.9874,0.9874
4,6990,12175934,2014-04-25,"Being a first time Airbnb user, I had a hard t...",Manhattan,Private room,70.0,186,Positive,0.0112,0.0774,0.9114,0.9114


In [ ]:

# ============================================================
# 6. FULLRUN 
# ============================================================

# Change batch size and run on full samples
BATCH_SIZE = 32  # GPU

manhattan_transformer = run_transformer_analysis(manhattan_sample, 'Manhattan', batch_size=BATCH_SIZE)
brooklyn_transformer  = run_transformer_analysis(brooklyn_sample,  'Brooklyn',  batch_size=BATCH_SIZE)

# Save
os.makedirs("data/processed/transformer_results", exist_ok=True)
manhattan_transformer.to_csv("data/processed/transformer_results/manhattan_transformer.csv", index=False)
brooklyn_transformer.to_csv("data/processed/transformer_results/brooklyn_transformer.csv",   index=False)